# RK1828 Qwen3-ASR Speech Recognition Demo


[Qwen3-ASR](https://github.com/QwenLM/Qwen3-ASR) (Tongyi Qianwen) supports
**52 languages and dialects** with automatic language detection. Split into two
parts, both on the NPU:

```
audio(wav) -> [Audio Encoder] features -> [LLM] autoregressive text -> transcript
              encoder.rknn (fp16)       llm.rknn (W4A16 quantized)
```

| Step | Content |
|---|---|
| 1 | Download models and sample audio (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Pick an audio clip and listen |
| 4 | Transcribe |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.


Sample audio comes from the other ASR repos, shared under `datasets/asr/`.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("qwen3-asr-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_asr/demo_x86_64.tar.gz"),
    ("qwen3-asr-rknn3-rk1828", "mel_128_filters.txt", "demos/qwen3_asr/mel_128_filters.txt"),
    ("qwen3-asr-rknn3-rk1828", "model/encoder.rknn", "demos/qwen3_asr/model/encoder.rknn"),
    ("qwen3-asr-rknn3-rk1828", "model/encoder.weight", "demos/qwen3_asr/model/encoder.weight"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.embed.bin", "demos/qwen3_asr/model/llm.embed.bin"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.rknn", "demos/qwen3_asr/model/llm.rknn"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.tokenizer.gguf", "demos/qwen3_asr/model/llm.tokenizer.gguf"),
    ("qwen3-asr-rknn3-rk1828", "model/llm.weight", "demos/qwen3_asr/model/llm.weight"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_asr/demo_x86_64.tar.gz", "demos/qwen3_asr")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Pick an audio clip and listen

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="Audio:", layout=widgets.Layout(width="60%"))
display(w_wav)
if any(o.endswith('/test_zh.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/test_zh.wav')][0]
print("Then keep running downwards ▼")

Listen to the audio first:

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. Transcribe

`0xff 0xff` are device/session placeholders. An 8.9 s Chinese clip was
transcribed fully and correctly in our tests.

In [ ]:
import os

DEMO = "demos/qwen3_asr"
wav = os.path.abspath(w_wav.value)
run_demo(["./rknn_qwen3_asr_demo",
          "model/encoder.rknn", "model/encoder.weight",
          "model/llm.rknn", "model/llm.weight",
          "model/llm.tokenizer.gguf", "model/llm.embed.bin",
          "0xff", "0xff", wav], DEMO)

## Summary

- Encoder + LLM decoupled: acoustic features and text generation live in two
  separate RKNN models.
- The tarball also ships `rknn_qwen3_asr_demo_online` (streaming) — give it a try.